In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Load dataset
df = pd.read_csv("LoanApprovalPrediction.csv")

# 1. Financial Domain Feature Engineering
df["Total_Income"] = df["ApplicantIncome"] + df["CoapplicantIncome"]
df["Loan_to_Income_Ratio"] = (df["LoanAmount"] * 1000) / (
    df["Total_Income"] + 1
)

df["Estimated_EMI"] = (df["LoanAmount"] * 1000) / df[
    "Loan_Amount_Term"
].replace(0, np.nan)
df["Annuity_to_Income_Ratio"] = df["Estimated_EMI"] / (
    (df["Total_Income"] / 12) + 1
)

df["Dependents_Clean"] = (
    df["Dependents"].replace("3+", 3).fillna(0).astype(float)
)
df["Income_Per_Capita"] = df["Total_Income"] / (df["Dependents_Clean"] + 1)

# 2. Features and Target Definition
numeric_features = [
    "ApplicantIncome",
    "CoapplicantIncome",
    "LoanAmount",
    "Total_Income",
    "Loan_to_Income_Ratio",
    "Estimated_EMI",
    "Annuity_to_Income_Ratio",
    "Income_Per_Capita",
]

categorical_features = [
    "Gender",
    "Married",
    "Education",
    "Self_Employed",
    "Property_Area",
    "Credit_History",
]

# 3. Leak-free Transformers
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# 4. Assembled Model Pipeline
full_model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=150,
                max_depth=6,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

# 5. Split and Train
X = df[numeric_features + categorical_features]
y = df["Loan_Status"].map({"Y": 1, "N": 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

full_model_pipeline.fit(X_train, y_train)

# 6. Save Single Unified Pipeline Artifact
joblib.dump(full_model_pipeline, "credit_underwriting_pipeline.pkl")
print(
    "Successfully generated and exported: credit_underwriting_pipeline.pkl"
)

Successfully generated and exported: credit_underwriting_pipeline.pkl
